# 02 · Preprocesamiento y Feature Engineering Temporal

Objetivo (PDF sección 3.2): generar variables temporales (ventanas, rezagos, agregaciones por periodo) y definir el split train/val/test respetando estrictamente el orden cronológico.

Decisiones de esta etapa, basadas en los hallazgos del notebook 01 (EDA temporal).

In [1]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd

from data import load_transaction_data
from features import add_time_features, add_rolling_user_features, chronological_split

df = load_transaction_data()
df = add_time_features(df)
print(df.shape)

(590540, 437)


## 1. Estrategia de valores faltantes

El notebook 01 mostró que solo 24 de 438 columnas no tienen nulos, y que las columnas de identity (id_01 a id_38, DeviceType, DeviceInfo) tienen una cobertura muy inestable en el tiempo, entre 16% y 55%. Con eso en mente, se decidió lo siguiente:

- Las columnas de identity se descartan una por una y se reemplazan por un solo flag, has_identity, que indica si la transacción tenía o no datos de identidad. Mantener 41 columnas con más de 90% de nulos, que además varían mucho en el tiempo, no aporta una señal confiable y complica el pipeline sin necesidad.
- Las variables numéricas restantes (C1 a C14, D1 a D15, V1 a V339, dist1, dist2) se imputan con -999, una constante fuera de rango que no es una estadística aprendida. Como no depende del split, no hay riesgo de fuga de información.
- Las variables categóricas (ProductCD, card4, card6, P_emaildomain, R_emaildomain, M1 a M9) tratan los nulos como una categoría más, missing, y se codifican a enteros. El mapeo de categoría a código es un vocabulario fijo, no una estadística del target, así que tampoco hay fuga al calcularlo antes del split.

In [2]:
identity_cols = [c for c in df.columns if c.startswith('id_') or c in ('DeviceType', 'DeviceInfo')]
df['has_identity'] = df['id_01'].notna().astype(int)
df = df.drop(columns=identity_cols)
print('columnas de identity descartadas:', len(identity_cols))
print('shape tras descartar identity:', df.shape)

categorical_cols = df.select_dtypes(include='object').columns.tolist()
print('columnas categoricas:', categorical_cols)

for col in categorical_cols:
    df[col] = df[col].fillna('missing').astype('category').cat.codes

numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if c not in ('TransactionID', 'isFraud')]
n_before = df[numeric_cols].isna().sum().sum()
df[numeric_cols] = df[numeric_cols].fillna(-999)
print('valores numericos nulos imputados con -999:', int(n_before))
print('nulos restantes en todo el dataframe:', int(df.isna().sum().sum()))

columnas de identity descartadas: 40
shape tras descartar identity: (590540, 398)
columnas categoricas: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']
valores numericos nulos imputados con -999: 92362478
nulos restantes en todo el dataframe: 0


El dataframe queda sin las columnas individuales de identity, con has_identity como señal resumen, sin nulos en general, y con las categóricas ya codificadas a enteros.

## 2. Tratamiento de outliers en TransactionAmt

El notebook 01 encontró un par de transacciones atípicas de $31,937.39. Se winsoriza TransactionAmt al percentil 99.5, para que un puñado de outliers no termine dominando el entrenamiento de modelos sensibles a escala como la Regresión Logística, y se agrega log_amt (log1p de TransactionAmt) como feature adicional, útil para los modelos lineales porque el monto tiene una distribución bastante sesgada.

In [3]:
p995 = df['TransactionAmt'].quantile(0.995)
print('percentil 99.5 de TransactionAmt:', round(p995, 2))
print('transacciones por encima del percentil 99.5:', (df['TransactionAmt'] > p995).sum())

df['TransactionAmt'] = df['TransactionAmt'].clip(upper=p995)
df['log_amt'] = np.log1p(df['TransactionAmt'])
print(df[['TransactionAmt', 'log_amt']].describe())

percentil 99.5 de TransactionAmt: 1656.11
transacciones por encima del percentil 99.5: 2952
       TransactionAmt        log_amt
count   590540.000000  590540.000000
mean       131.381798       4.381313
std        196.398197       0.931396
min          0.251000       0.223943
25%         43.321000       3.791459
50%         68.769000       4.245190
75%        125.000000       4.836282
max       1656.110000       7.412830


## 3. Variables agregadas por ventana temporal

Se agrega, por card1 como proxy de tarjeta o usuario, el conteo y el monto promedio de transacciones en las 24 horas previas. Es justo el tipo de variable rezagada o agregada por periodo que pide el PDF en la sección 3.2, y como solo mira hacia atrás en el tiempo, es segura de calcular antes del split.

In [4]:
df = add_rolling_user_features(df, user_key='card1', window_seconds=86400)
new_cols = [c for c in df.columns if c.startswith('card1_')]
print('nuevas columnas:', new_cols)
print(df[new_cols].describe())

nuevas columnas: ['card1_txn_count_86400s', 'card1_amt_mean_86400s']
       card1_txn_count_86400s  card1_amt_mean_86400s
count           590540.000000          590540.000000
mean                19.803627             131.424886
std                 45.610592             134.380204
min                  1.000000               0.251000
25%                  2.000000              59.000000
50%                  6.000000             100.306458
75%                 20.000000             150.815705
max                881.000000            1656.110000


## 4. Split cronológico estricto (train, val y test)

60%, 20% y 20%, respetando el orden temporal y sin aleatorizar, tal como exige la restricción de consistencia temporal estricta del checkpoint 1 y el PDF del proyecto, que pide evitar supuestos de independencia y de distribución idéntica.

In [5]:
train_df, val_df, test_df = chronological_split(df, dt_col='TransactionDT', train_frac=0.6, val_frac=0.2)

for name, part in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'{name}: shape={part.shape}, dias={part["day"].min()}-{part["day"].max()}, '
          f'tasa_fraude={part["isFraud"].mean():.4f}')

train: shape=(354324, 401), dias=0-100, tasa_fraude=0.0338
val: shape=(118108, 401), dias=100-140, tasa_fraude=0.0390
test: shape=(118108, 401), dias=140-181, tasa_fraude=0.0344


## 5. Guardado de los datasets procesados

Se guardan en data/processed/ en formato Parquet, que es más eficiente que CSV para reutilizarlos en los notebooks 03 a 05, dado el tamaño del dataset.

In [6]:
import os
os.makedirs('../data/processed', exist_ok=True)

train_df.to_parquet('../data/processed/train.parquet', index=False)
val_df.to_parquet('../data/processed/val.parquet', index=False)
test_df.to_parquet('../data/processed/test.parquet', index=False)

for name in ['train', 'val', 'test']:
    size_mb = os.path.getsize(f'../data/processed/{name}.parquet') / 1e6
    print(f'{name}.parquet: {size_mb:.1f} MB')

train.parquet: 57.9 MB
val.parquet: 18.7 MB
test.parquet: 19.9 MB


## Conclusiones para el notebook 03

- El dataframe queda sin nulos, con las categóricas codificadas, y listo para los 4 modelos candidatos sin pasos adicionales de preprocesamiento.
- has_identity, log_amt, hour_of_day, day_of_week y las variables card1_txn_count_86400s y card1_amt_mean_86400s son las features nuevas respecto al dataset crudo.
- El split respeta estrictamente el orden temporal. El notebook 03 debe entrenar solo con train.parquet, elegir hiperparámetros con val.parquet, y reportar la métrica final una sola vez sobre test.parquet.
- Conviene revisar la tasa de fraude por split (ver sección 4) al interpretar los resultados. Como no es un split aleatorio, no está garantizado que sea igual entre train, val y test, y eso es justamente lo que se espera en un problema no estacionario.